# BIG DATA: PRÉ-PROCESSAMENTO EM LARGA ESCALA, HARDWARE E FORMATO PARQUET


## Situação de Aprendizagem — O Desafio do Novo Bolsa Família na *Data Organized*

Na Aula 11, aprendemos os **5 Vs do Big Data**, a divisão entre o **Engenheiro de Dados** e o **Cientista de Dados**, e vimos como o **Polars** utiliza o motor em Rust e a memória colunar *Apache Arrow* para processar milhões de linhas sem engasgar.

Hoje avançamos para o **coração da engenharia de preparação de dados** na **Data Organized**:
O cliente nos entregou os arquivos mensais da base pública do **Novo Bolsa Família** (Portal da Transparência). Na vida real, cada arquivo mensal `.csv` pode ter entre **1,5 GB e 3 GB** (somando até 14 GB em 5 meses). Para o nosso laboratório de hoje, vamos trabalhar com uma amostra de **3 meses** (`Janeiro`, `Fevereiro` e `Março`) que somam **aproximadamente 200 MB** (**2.886.000 registros**).

O Analista Sênior reforçou a regra de ouro do projeto:
"Nunca faça análises diretamente sobre arquivos `.csv` gigantes! Primeiro executamos o **Pré-processamento** (ler os CSVs brutos, tratar textos monetários com vírgula, unificar os meses e compactar tudo em um arquivo **`.parquet`**). Depois, na fase de **Processamento**, qualquer consulta estatística levará frações de segundo."

### Bibliotecas Utilizadas

*Instalação (caso ainda não estejam instaladas no seu ambiente virtual):*

```bash
pip install polars pandas numpy pyarrow psutil
```

## Bloco 1: Por que Separar o Pré-processamento do Processamento?

### Analogia da Cozinha Profissional (*Mise en Place*)
Imagine uma cozinha industrial que precisa servir 500 pratos na hora do almoço:
- **Etapa 1 — Pré-processamento (*Mise en place*):** É o trabalho pesado feito pela manhã: lavar, descascar, cortar os legumes, temperar e guardar tudo organizado em potes transparentes e etiquetados. Você faz isso **uma única vez**.
- **Etapa 2 — Processamento (Preparo e Serviço):** Quando os pedidos chegam, o cozinheiro apenas abre os potes prontos e finaliza o prato em segundos, sem perder tempo descascando batatas a cada pedido.

### O que acontece dentro do Pipeline de Big Data:
| Fase do Pipeline | O que fazemos na prática? | Por que separar? |
| :--- | :--- | :--- |
| **1. Pré-processamento (Aula 12)** | Ler múltiplos `.csv` brutos, corrigir separadores (`;`), converter textos com vírgula (`'650,00'`) em números (`650.00`), concatenar os meses e salvar em **`.parquet`**. | Evita repetir o trabalho lento de leitura e limpeza de texto toda vez que você rodar um cálculo. |
| **2. Processamento (Aula 13)** | Ler o `.parquet` pronto para calcular médias, medianas, quartis, outliers e construir gráficos. | Permite testar dezenas de hipóteses estatísticas em segundos sem travar a RAM. |

## Bloco 2: Anatomia Interna do Apache Parquet (`.parquet`)

### Analogia do Dia a Dia: Mala Bagunçada vs. Roupa Embalada a Vácuo
- **Arquivo `.csv` (Orientado a Linhas / Texto Puro):** É como jogar todas as roupas soltas dentro de uma mala gigante. Cada número é gravado como texto caractere por caractere, repetindo vírgulas e cabeçalhos milhões de vezes. Para ler uma única coluna, o computador é obrigado a ler a linha inteira.
- **Arquivo `.parquet` (Binário Orientado a Colunas — Apache Parquet):** É como usar **sacos organizadores embalados a vácuo**! Cada coluna é armazenada separadamente em blocos binários altamente compactados.

### Como o `.parquet` consegue reduzir até 80% do tamanho no disco?
1. **Codificação por Dicionário (*Dictionary Encoding*):** Se a coluna `UF` tem 3 milhões de linhas, mas apenas 27 siglas diferentes (`'RJ'`, `'SP'`, `'BA'`...), o Parquet cria um dicionário interno (`0 = 'RJ'`, `1 = 'SP'`) e grava apenas números minúsculos no lugar de repetir textos longos.
2. **Codificação por Repetição (*Run-Length Encoding - RLE*):** Se o valor `'202601'` aparece 962.000 vezes seguidas na coluna `MES_COMPETENCIA`, o Parquet não grava `'202601'` quase um milhão de vezes — ele grava apenas: *"repita `'202601'` 962.000 vezes"*!
3. **Compressão Binária (*Snappy / Zstd*):** Após organizar as colunas, aplica algoritmos ultrarrápidos de compressão.
4. **Preservação de Esquema (*Schema*):** Grava no cabeçalho o tipo exato de cada coluna (`Int64`, `Float64`, `String`), garantindo que números nunca voltem como texto.
5. **Poda de Colunas (*Column Pruning*) e Metadados (Min/Max):** Permite ler apenas as colunas pedidas e pular blocos inteiros usando os valores mínimos e máximos gravados nos metadados.

## Bloco 3: Dicionário de Dados da Base (Novo Bolsa Família — Amostra 3 Meses)

Antes de escrever qualquer linha de código, todo analista profissional estuda o **Dicionário de Dados** para conhecer o significado, o formato e as armadilhas de cada coluna do CSV bruto:

| # | Nome da Coluna | Tipo Bruto no CSV | Tipo Após Limpeza | Descrição Detalhada e Regra de Negócio |
| :-: | :--- | :--- | :--- | :--- |
| **1** | `MES_COMPETENCIA` | Texto (`String`) | Texto (`String`) | Ano e mês de competência da folha de pagamento no formato `AAAAMM` (ex.: `'202601'`, `'202602'`, `'202603'`). |
| **2** | `MES_REFERENCIA` | Texto (`String`) | Texto (`String`) | Ano e mês de referência utilizado para apuração do benefício (`AAAAMM`). |
| **3** | `UF` | Texto (`String`) | Texto (`String`) | Sigla da Unidade da Federação (Estado) onde reside a família beneficiária (ex.: `'RJ'`, `'SP'`, `'BA'`). |
| **4** | `CODIGO_MUNICIPIO_SIAFI` | Inteiro (`Int64`) | Inteiro (`Int64`) | Código numérico oficial do município no sistema SIAFI do Governo Federal. |
| **5** | `NOME_MUNICIPIO` | Texto (`String`) | Texto (`String`) | Nome oficial do município em letras maiúsculas sem acentuação (ex.: `'RIO DE JANEIRO'`). |
| **6** | `NIS_FAVORECIDO` | Inteiro (`Int64`) | Inteiro (`Int64`) | Número de Identificação Social (11 dígitos) do responsável familiar titular do benefício. |
| **7** | `NOME_FAVORECIDO` | Texto (`String`) | Texto (`String`) | Nome completo do titular que recebe o benefício social. |
| **8** | `VALOR_PARCELA` | **Texto (`String`)** | **Decimal (`Float64`)** | Valor financeiro pago no mês em Reais. **Atenção:** No CSV bruto vem como texto com vírgula (ex.: `'650,00'`) e precisa ser convertido para `Float64` (`650.00`) antes de calcular médias ou salvar em Parquet! |

## Bloco 4: Gestão de Hardware na Estação de Trabalho

Ao manipular arquivos que somam **~200 MB no disco** (e que expandem na memória RAM durante a leitura), siga estas boas práticas do nosso Plano de Curso:
1. **Feche programas concorrentes:** Navegadores com muitas abas, Teams e editores pesados disputam memória RAM e núcleos da CPU com o Python.
2. **Acompanhe pelo Gerenciador de Tarefas (`Ctrl + Shift + Esc` $\rightarrow$ Aba Desempenho):**
   - **Memória:** Se ultrapassar 90%, o Windows ativa o *Swap* (memória virtual no SSD), derrubando a velocidade.
   - **CPU (Processadores Lógicos):** Observe todos os núcleos trabalhando em paralelo no Polars.
3. **Limpeza explícita de memória (`del` + `gc.collect()`):** Sempre que terminar de usar um DataFrame pesado que não será mais necessário, apague-o da memória RAM.

## Exemplo 1 (Nível Básico): Gerando os 3 Arquivos Mensais (~200 MB Totais) e Inspecionando o Schema

Vamos gerar **3 arquivos CSV mensais** (`bolsa_familia_202601.csv`, `bolsa_familia_202602.csv` e `bolsa_familia_202603.csv`) com **962.000 linhas cada** (**2.886.000 registros no total**), calibrados para ocupar **aproximadamente 200 MB** no disco (~66,7 MB por arquivo), seguindo fielmente o Dicionário de Dados acima.

In [1]:
# Importa o módulo 'os' para consultar o tamanho exato dos arquivos gravados no disco
import os

# Importa o módulo 'time' para cronometrar o tempo de cada operação em segundos
import time

# Importa o módulo 'gc' (Garbage Collector) para forçar a liberação de memória RAM
import gc

# Importa NumPy para gerar os vetores de dados simulados em alta velocidade
import numpy as np

# Importa Pandas e Polars para compararmos performance, memória e fluxo de trabalho
import pandas as pd
import polars as pl

# Fixa a semente aleatória (42) para que os números gerados sejam idênticos em toda a turma
np.random.seed(42)

# Define 962.000 registros por mês (3 meses x 962.000 = 2.886.000 linhas ≈ 200 MB em CSV)
N_LINHAS_MES = 962_000

# Lista com os códigos dos 3 meses do 1º trimestre de 2026
meses_trimestre = ['202601', '202602', '202603']

# Lista para guardar os nomes dos 3 arquivos CSV criados
arquivos_csv = []

# Loop que gera e salva cada um dos 3 arquivos mensais no padrão do Portal da Transparência
for mes in meses_trimestre:
    nome_arquivo = f"bolsa_familia_{mes}.csv"                                                       # Define o nome do arquivo mensal
    valores_float = np.round(np.random.uniform(600.0, 1150.0, size=N_LINHAS_MES), 2)                # Sorteia valores entre R$ 600 e R$ 1150
    valores_brutos_texto = [f"{v:.2f}".replace('.', ',') for v in valores_float]                    # Converte para texto com vírgula ('650,00')
    
    # Monta o DataFrame com as 8 colunas descritas no Dicionário de Dados
    df_mes = pl.DataFrame({
        'MES_COMPETENCIA': np.full(N_LINHAS_MES, mes),                                              # Coluna 1: Competência AAAAMM
        'MES_REFERENCIA': np.full(N_LINHAS_MES, mes),                                               # Coluna 2: Referência AAAAMM
        'UF': np.random.choice(['RJ', 'SP', 'MG', 'BA', 'PE', 'CE', 'PA', 'MA', 'PR', 'RS'], size=N_LINHAS_MES), # Coluna 3: Estado
        'CODIGO_MUNICIPIO_SIAFI': np.random.randint(1000, 9999, size=N_LINHAS_MES, dtype=np.int64), # Coluna 4: Código SIAFI
        'NOME_MUNICIPIO': np.random.choice(['RIO DE JANEIRO', 'SAO PAULO', 'BELO HORIZONTE', 'SALVADOR', 'FORTALEZA', 'RECIFE', 'CURITIBA', 'PORTO ALEGRE'], size=N_LINHAS_MES), # Coluna 5: Município
        'NIS_FAVORECIDO': np.random.randint(10000000000, 99999999999, size=N_LINHAS_MES, dtype=np.int64), # Coluna 6: NIS 11 dígitos
        'NOME_FAVORECIDO': np.random.choice(['MARIA DA SILVA SANTOS', 'JOSE OLIVEIRA SOUZA', 'ANA PAULA FERREIRA', 'CARLOS EDUARDO LIMA', 'FRANCISCA DAS CHAGAS'], size=N_LINHAS_MES), # Coluna 7: Nome
        'VALOR_PARCELA': valores_brutos_texto                                                       # Coluna 8: Valor em texto com vírgula
    })
    
    # Grava o arquivo no disco usando ponto e vírgula (';') como separador
    df_mes.write_csv(nome_arquivo, separator=';')
    arquivos_csv.append(nome_arquivo)
    
    # Apaga o DataFrame temporário da memória RAM antes de gerar o próximo mês
    del df_mes, valores_float, valores_brutos_texto
    gc.collect()

# Calcula o tamanho individual e a soma total em Megabytes (MB) dos 3 arquivos no disco
soma_total_csv_mb = 0.0
for arq in arquivos_csv:
    tam_mb = os.path.getsize(arq) / (1024 ** 2)                                                     # Converte bytes para MB
    soma_total_csv_mb += tam_mb
    print(f"Arquivo gerado: {arq:28s} | Tamanho: {tam_mb:.2f} MB")

print("-" * 65)
print(f"Tamanho TOTAL somado dos 3 meses em CSV: {soma_total_csv_mb:.2f} MB (2.886.000 linhas)")

Arquivo gerado: bolsa_familia_202601.csv     | Tamanho: 66.67 MB
Arquivo gerado: bolsa_familia_202602.csv     | Tamanho: 66.67 MB
Arquivo gerado: bolsa_familia_202603.csv     | Tamanho: 66.67 MB
-----------------------------------------------------------------
Tamanho TOTAL somado dos 3 meses em CSV: 200.01 MB (2.886.000 linhas)


*Leitura Analítica (Exemplo 1):*

Criamos uma amostra realista de **3 meses** (`2.886.000` registros) ocupando **~200 MB** no disco, inclusive reproduzindo o problema clássico das bases governamentais: a coluna `VALOR_PARCELA` gravada como texto com vírgula (`'650,00'`).

## Exemplo 2 (Nível Básico-Intermediário): Duelo de Leitura e Concatenação — Pandas vs. Polars (200 MB)

Agora vamos ler os **3 arquivos CSV (~200 MB)** e empilhá-los (`concat`) primeiro com **Pandas** e depois com **Polars**, medindo o tempo e a memória RAM consumida por cada biblioteca.

In [3]:
# ==============================================================================
# TESTE A: LENDO E CONCATENANDO OS 3 MESES (~200 MB) COM PANDAS
# ==============================================================================

# Marca o tempo inicial antes da leitura com Pandas
t0_pd = time.time()

# Lê cada um dos 3 arquivos CSV separados por ';' usando list comprehension
lista_pd = [pd.read_csv(arq, sep=';') for arq in arquivos_csv]

# Empilha os 3 DataFrames em uma única tabela de 2.886.000 linhas
df_pd_3m = pd.concat(lista_pd, ignore_index=True)

# Calcula o tempo total gasto pelo Pandas e a memória RAM ocupada pelo DataFrame
tempo_pd_3m = time.time() - t0_pd
ram_pd_3m_mb = df_pd_3m.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"[PANDAS] Linhas: {len(df_pd_3m):,} | Tempo: {tempo_pd_3m:.2f}s | RAM ocupada: {ram_pd_3m_mb:.1f} MB")

# Libera toda a memória usada pelo Pandas antes de iniciar o teste com Polars
del lista_pd, df_pd_3m
gc.collect()

# ==============================================================================
# TESTE B: LENDO E CONCATENANDO OS 3 MESES (~200 MB) COM POLARS
# ==============================================================================

# Marca o tempo inicial antes da leitura com Polars
t0_pl = time.time()

# Lê cada um dos 3 arquivos CSV em paralelo usando todos os núcleos da CPU
lista_pl = [pl.read_csv(arq, separator=';') for arq in arquivos_csv]

# Empilha os 3 DataFrames na memória colunar Apache Arrow
df_pl_3m = pl.concat(lista_pl)

# Calcula o tempo total gasto pelo Polars e a memória RAM ocupada
tempo_pl_3m = time.time() - t0_pl
ram_pl_3m_mb = df_pl_3m.estimated_size('mb')

print(f"[POLARS] Linhas: {len(df_pl_3m):,} | Tempo: {tempo_pl_3m:.2f}s | RAM ocupada: {ram_pl_3m_mb:.1f} MB")

print("-" * 70)
print(f"GANHO DE VELOCIDADE: Polars foi {tempo_pd_3m / tempo_pl_3m:.1f}x mais rápido na leitura!")
print(f"ECONOMIA DE RAM:     Polars usou {ram_pd_3m_mb / ram_pl_3m_mb:.1f}x menos memória RAM!")

# Exibe as 3 primeiras linhas e o Schema (tipos das colunas) carregados pelo Polars
print("\nAmostra dos dados brutos carregados:")
print(df_pl_3m.head(3))

[PANDAS] Linhas: 2,886,000 | Tempo: 0.25s | RAM ocupada: 279.8 MB
[POLARS] Linhas: 2,886,000 | Tempo: 0.18s | RAM ocupada: 191.8 MB
----------------------------------------------------------------------
GANHO DE VELOCIDADE: Polars foi 1.3x mais rápido na leitura!
ECONOMIA DE RAM:     Polars usou 1.5x menos memória RAM!

Amostra dos dados brutos carregados:
shape: (3, 8)
┌─────────────┬─────────────┬─────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ MES_COMPETE ┆ MES_REFEREN ┆ UF  ┆ CODIGO_MUN ┆ NOME_MUNIC ┆ NIS_FAVORE ┆ NOME_FAVOR ┆ VALOR_PARC │
│ NCIA        ┆ CIA         ┆ --- ┆ ICIPIO_SIA ┆ IPIO       ┆ CIDO       ┆ ECIDO      ┆ ELA        │
│ ---         ┆ ---         ┆ str ┆ FI         ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ i64         ┆ i64         ┆     ┆ ---        ┆ str        ┆ i64        ┆ str        ┆ str        │
│             ┆             ┆     ┆ i64        ┆            ┆            ┆            ┆            │
╞═════════════╪══════

*Leitura Analítica (Exemplo 2):*

Observe o cabeçalho da coluna `VALOR_PARCELA`: ela foi carregada como **`str` (texto)** por causa da vírgula decimal (`'650,00'`). Se tentarmos calcular `.mean()` ou `.sum()` agora, dará erro! Precisamos limpar e tipar essa coluna antes de exportar para `.parquet`.

## Exemplo 3 (Nível Intermediário): Limpeza de Schema e Compactação para `.parquet`

Agora vamos executar o fluxo completo de **Pré-processamento Profissional**:
1. Converter `VALOR_PARCELA` de texto (`'650,00'`) para número decimal (`Float64`).
2. Exportar os **2.886.000 registros** para um único arquivo `'bolsa_familia_3meses.parquet'`.
3. Comparar o tamanho do `.parquet` no disco contra os **~200 MB** dos 3 CSVs originais e medir a velocidade de leitura do `.parquet`.

In [4]:
# Passo 1: Limpa a coluna VALOR_PARCELA (troca ',' por '.' e converte de String para Float64)
df_pl_limpo = df_pl_3m.with_columns(
    pl.col('VALOR_PARCELA')                          # Seleciona a coluna de texto
    .str.replace(',', '.')                           # Substitui a vírgula decimal por ponto
    .cast(pl.Float64)                                # Converte o tipo para número decimal de 64 bits
    .alias('VALOR_PARCELA')                          # Sobrescreve a própria coluna com o tipo corrigido
)

# Libera o DataFrame bruto antigo da memória RAM
del lista_pl, df_pl_3m
gc.collect()

# Passo 2: Exporta (compacta) o DataFrame limpo para o formato Apache Parquet
t0_export = time.time()
df_pl_limpo[['MES_COMPETENCIA', 'MES_REFERENCIA', 'UF', 'CODIGO_MUNICIPIO_SIAFI', 'NOME_MUNICIPIO', 'NIS_FAVORECIDO', 'VALOR_PARCELA']].write_parquet('bolsa_familia_3meses.parquet')   # Grava no disco com compressão colunar
tempo_export_pq = time.time() - t0_export

# Passo 3: Mede o tamanho do arquivo .parquet gerado no disco e calcula a redução percentual
tam_pq_mb = os.path.getsize('bolsa_familia_3meses.parquet') / (1024 ** 2)
reducao_disco = (1 - (tam_pq_mb / soma_total_csv_mb)) * 100

# Passo 4: Cronometra a leitura do arquivo .parquet pronto do disco
t0_read_pq = time.time()
df_parquet_pronto = pl.read_parquet('bolsa_familia_3meses.parquet')
tempo_read_pq = time.time() - t0_read_pq

# Exibe o relatório completo de compactação e ganho de tempo
print(f"1. Tempo para compactar e salvar o Parquet: {tempo_export_pq:.2f} segundos")
print(f"-" * 70)
print(f"2. Espaço dos 3 CSVs originais no disco:    {soma_total_csv_mb:.2f} MB")
print(f"   Espaço do arquivo Parquet no disco:      {tam_pq_mb:.2f} MB")
print(f"   --> Redução de espaço em disco:          {reducao_disco:.1f}% MENOR!")
print(f"-" * 70)
print(f"3. Tempo de leitura dos 3 CSVs (Polars):    {tempo_pl_3m:.2f} segundos")
print(f"   Tempo de leitura do Parquet (Polars):    {tempo_read_pq:.4f} segundos")
print(f"   --> Ler o Parquet foi {tempo_pl_3m / tempo_read_pq:.1f}x MAIS RÁPIDO que ler os CSVs!")

1. Tempo para compactar e salvar o Parquet: 0.13 segundos
----------------------------------------------------------------------
2. Espaço dos 3 CSVs originais no disco:    200.01 MB
   Espaço do arquivo Parquet no disco:      31.31 MB
   --> Redução de espaço em disco:          84.3% MENOR!
----------------------------------------------------------------------
3. Tempo de leitura dos 3 CSVs (Polars):    0.18 segundos
   Tempo de leitura do Parquet (Polars):    0.0902 segundos
   --> Ler o Parquet foi 2.0x MAIS RÁPIDO que ler os CSVs!


*Leitura Analítica (Exemplo 3):*

Olhe a transformação: de **~200 MB** espalhados em 3 arquivos CSV lentos, passamos para **um único arquivo `.parquet` de ~30 MB** (uma redução superior a **80% no disco**!) que pode ser lido integralmente em **décimos de segundo** e já vem com `VALOR_PARCELA` tipado como `Float64`.

## Exemplo 4 (Nível Intermediário): Poda de Colunas (*Column Pruning*) na Leitura do `.parquet`

Como o `.parquet` guarda cada coluna separadamente, se quisermos calcular a média de pagamentos por estado (`UF`), não precisamos carregar os nomes das pessoas (`NOME_FAVORECIDO`) nem o código SIAFI na RAM — basta informar `columns=['UF', 'VALOR_PARCELA']`.

In [6]:
# Mede a RAM ocupada quando carregamos todas as 8 colunas do Parquet
ram_8_colunas_mb = df_parquet_pronto.estimated_size('mb')

# Lê APENAS as 2 colunas necessárias para análise estatística por estado (Column Pruning)
df_enxuto = pl.read_parquet(
    'bolsa_familia_3meses.parquet',
    columns=['UF', 'VALOR_PARCELA']                                 # Ignora as outras 6 colunas direto no disco!
)
ram_2_colunas_mb = df_enxuto.estimated_size('mb')

# Calcula a soma e a mediana por estado sobre os 2,88 milhões de registros
resumo_uf = (
    df_enxuto
    .group_by('UF')
    .agg([
        pl.col('VALOR_PARCELA').count().alias('Qtd_Beneficios'),
        pl.col('VALOR_PARCELA').median().alias('Mediana_BRL'),
        pl.col('VALOR_PARCELA').mean().alias('Media_BRL'),
        pl.col('VALOR_PARCELA').sum().alias('Total_Pago_BRL')
    ])
    .sort('Total_Pago_BRL', descending=True)
)

print(f"RAM com todas as 8 colunas: {ram_8_colunas_mb:.2f} MB")
print(f"RAM com apenas 2 colunas:   {ram_2_colunas_mb:.2f} MB (Economia de {(1 - ram_2_colunas_mb/ram_8_colunas_mb)*100:.1f}% de RAM!)\n")
print(resumo_uf)

RAM com todas as 8 colunas: 143.11 MB
RAM com apenas 2 colunas:   27.52 MB (Economia de 80.8% de RAM!)

shape: (10, 5)
┌─────┬────────────────┬─────────────┬────────────┬────────────────┐
│ UF  ┆ Qtd_Beneficios ┆ Mediana_BRL ┆ Media_BRL  ┆ Total_Pago_BRL │
│ --- ┆ ---            ┆ ---         ┆ ---        ┆ ---            │
│ str ┆ u32            ┆ f64         ┆ f64        ┆ f64            │
╞═════╪════════════════╪═════════════╪════════════╪════════════════╡
│ PR  ┆ 289527         ┆ 875.02      ┆ 875.060883 ┆ 2.5335e8       │
│ BA  ┆ 289222         ┆ 874.84      ┆ 874.619176 ┆ 2.5296e8       │
│ PA  ┆ 288854         ┆ 875.81      ┆ 875.229566 ┆ 2.5281e8       │
│ RS  ┆ 288769         ┆ 875.48      ┆ 874.913463 ┆ 2.5265e8       │
│ CE  ┆ 288578         ┆ 875.165     ┆ 875.112717 ┆ 2.5254e8       │
│ SP  ┆ 288750         ┆ 874.22      ┆ 874.573407 ┆ 2.5253e8       │
│ RJ  ┆ 288307         ┆ 875.5       ┆ 875.355537 ┆ 2.5237e8       │
│ MA  ┆ 288236         ┆ 875.0       ┆ 875.029009 ┆ 2

*Leitura Analítica (Exemplo 4):*

Combinando **Parquet + Column Pruning**, reduzimos o uso de memória RAM em mais de **75%** e processamos quase 3 milhões de benefícios agrupados por estado instantaneamente.

---
## Atividades Práticas (Do Básico ao Intermediário)

### ATIVIDADE 1 (Nível Básico) — Limpeza de Schema e Seleção de Colunas
Utilizando o Dicionário de Dados da aula, crie um pequeno `pl.DataFrame` simulando 5 registros brutos do governo:
```python
df_teste_gov = pl.DataFrame({
    'MES_COMPETENCIA': ['202601', '202601', '202602', '202602', '202603'],
    'UF': ['RJ', 'SP', 'BA', 'MG', 'RJ'],
    'NOME_FAVORECIDO': ['ANA SILVA', 'JOAO SOUZA', 'MARIA LIMA', 'PEDRO COSTA', 'LUCIA DIAS'],
    'VALOR_PARCELA': ['600,00', '850,50', '700,00', '920,75', '650,00']
})
```

**Faça:**
1. Converta `VALOR_PARCELA` de texto com vírgula para número decimal (`Float64`) usando `.with_columns()` e `.str.replace(',', '.').cast(pl.Float64)`.
2. Remova a coluna `NOME_FAVORECIDO` (anonimização de dados pessoais — LGPD) usando `.drop('NOME_FAVORECIDO')`.
3. Salve o resultado em `'atividade1_anonimizada.parquet'` e leia-o de volta com `pl.read_parquet()`.

In [7]:
# Escreva aqui o código da Atividade 1 (Nível Básico):
from pathlib import Path

### Criação de DataFrame com 5 registros
df_teste_gov = pl.DataFrame({
    'MES_COMPETENCIA': ['202601', '202601', '202602', '202602', '202603'],
    'UF': ['RJ', 'SP', 'BA', 'MG', 'RJ'],
    'NOME_FAVORECIDO': ['ANA SILVA', 'JOAO SOUZA', 'MARIA LIMA', 'PEDRO COSTA', 'LUCIA DIAS'],
    'VALOR_PARCELA': ['600,00', '850,50', '700,00', '920,75', '650,00']
})

# Conversão do VALOR_PARCELA para número (atividade 1.1)
df = df_teste_gov.with_columns(
    pl.col('VALOR_PARCELA')                          # Seleciona a coluna de texto
    .str.replace(',', '.')                           # Substitui a vírgula decimal por ponto
    .cast(pl.Float64)                                # Converte o tipo para número decimal de 64 bits
    .alias('VALOR_PARCELA')                          # Sobrescreve a própria coluna com o tipo corrigido
)

# Remoção da coluna NOME_FAVORECIDO (atividade 1.2)
df.drop('NOME_FAVORECIDO')

# Salva o DataFrame em formato Parquet e lê o resultado final (atividade 1.3)`pl.read_parquet()`.
arquivo = Path('..') / 'Data' / 'atividade1_anonimizada.parquet'
df.write_parquet(arquivo)
pl.read_parquet(arquivo)

MES_COMPETENCIA,UF,NOME_FAVORECIDO,VALOR_PARCELA
str,str,str,f64
"""202601""","""RJ""","""ANA SILVA""",600.0
"""202601""","""SP""","""JOAO SOUZA""",850.5
"""202602""","""BA""","""MARIA LIMA""",700.0
"""202602""","""MG""","""PEDRO COSTA""",920.75
"""202603""","""RJ""","""LUCIA DIAS""",650.0


### ATIVIDADE 2 (Nível Intermediário) — Auditoria Regional sobre o Parquet Consolidado (2,88 Milhões de Linhas)
Usando o arquivo `'bolsa_familia_3meses.parquet'` que geramos no Exemplo 3 (com os 3 meses consolidados):

**Faça:**
1. Leia **apenas as colunas** `['MES_COMPETENCIA', 'UF', 'NOME_MUNICIPIO', 'VALOR_PARCELA']` usando `pl.read_parquet(..., columns=[...])`.
2. Filtre apenas os registros dos estados `'RJ'`, `'SP'` e `'BA'` (`pl.col('UF').is_in(['RJ', 'SP', 'BA'])`) onde `VALOR_PARCELA` seja maior que `R$ 800,00`.
3. Agrupe por `MES_COMPETENCIA` e `UF` e calcule: a **Quantidade de Pagamentos (`count`)**, a **Média (`mean`)**, a **Mediana (`median`)** e o **Desvio Padrão (`std(ddof=0)`)**.
4. Exporte essa tabela resumida final para `'relatorio_gerencial_trimestre.parquet'`.

In [ ]:
# Escreva aqui o código da Atividade 2 (Nível Intermediário):

# Lê colunas específicas do arquivo parquet (atividade 2.1)
df = pl.read_parquet(
    Path('..') / 'Data' / 'bolsa_familia_3meses.parquet',
    columns=['MES_COMPETENCIA', 'UF', 'NOME_MUNICIPIO', 'VALOR_PARCELA']
)

# Agrupa por 'MES_COMPETENCIA' e 'UF', calculando estatísticas (atividade 2.3)
df_enxuto = (
    df.filter((pl.col('VALOR_PARCELA') > 800) & (pl.col('UF').is_in(['SP', 'RJ', 'BA'])))  # Filtra parcelas maiores que R$800,00 apenas dos estados RJ, SP e BA (atividade 2.2)
    .group_by('MES_COMPETENCIA', 'UF')
    .agg([
        pl.col('VALOR_PARCELA').count().alias('Quantidade de Pagamentos'),
        pl.col('VALOR_PARCELA').median().alias('Mediana'),
        pl.col('VALOR_PARCELA').mean().alias('Média'),
        pl.col('VALOR_PARCELA').std(ddof=0).alias('Desvio Padrão')
    ])
    .sort(['MES_COMPETENCIA', 'UF'], descending=[False, False])
)

# Exporta e mostra o resultado em formato parquet (atividade 2.4)
arquivo = Path('..') / 'Data' / 'relatorio_gerencial_trimestre.parquet'
df_enxuto.write_parquet(arquivo)
pl.read_parquet(arquivo)

MES_COMPETENCIA,UF,Quantidade de Pagamentos,Mediana,Média,Desvio Padrão
i64,str,u32,f64,f64,f64
202601,"""BA""",61410,975.305,975.24998,100.98548
202601,"""RJ""",60773,975.55,975.473676,100.942573
202601,"""SP""",61111,973.98,974.50201,101.264304
202602,"""BA""",61085,974.42,974.411522,100.792326
202602,"""RJ""",61344,975.575,975.602461,101.116269
202602,"""SP""",60886,975.105,975.437538,101.035443
202603,"""BA""",61324,975.145,975.030499,100.987192
202603,"""RJ""",61497,974.24,974.600208,101.136676
202603,"""SP""",61551,973.69,973.915258,101.020982


*Leitura Analítica Final:*

Concluímos com rigor a etapa de **Pré-processamento em Big Data**: documentamos o **Dicionário de Dados**, tratamos tipos e dados sensíveis (LGPD), consolidamos **3 meses (~200 MB em CSV)** em um único arquivo **`.parquet` (~30 MB)** e aplicamos **Poda de Colunas**. Na **Aula 13**, vamos aprender o **Plano de Execução (*Lazy Evaluation*)** para processar milhões de linhas sem sequer carregar o arquivo inteiro na RAM!